# 06B — Nested Challenger Tuning, Calibration, and Threshold Policies

Exploratory post-freeze GPU analysis of XGBoost and CatBoost. The 05C LightGBM paper primary remains frozen. Hyperparameters, calibration methods, and thresholds are selected only from the frozen four inner incident folds inside each outer training set; each untouched outer fold is used once for evaluation.

Threshold policies are decision scenarios, not universal operating recommendations. `hold_label=1` means held; therefore false-positive hold predictions can be operationally consequential.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "06B_V2_GPU_NESTED_CHALLENGER_TUNING_CALIBRATION_THRESHOLDS"

from pathlib import Path
from datetime import datetime, timezone
import gc, importlib.util, json, subprocess, sys, time

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURES_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
INNER_FOLDS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "INNER_INCIDENT_FOLDS_LONG_V1.csv"
LIGHTGBM_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"
FIXED_ALTERNATIVES_PATH = PROJECT_ROOT / "data" / "processed" / "alternative_model_benchmark_v1" / "ALTERNATIVE_MODEL_ALL_OOF_PREDICTIONS_V1.parquet"
UPSTREAM_CHECKS_PATH = PROJECT_ROOT / "outputs" / "alternative_model_benchmark_v1" / "FINAL_ALTERNATIVE_MODEL_BENCHMARK_CHECKS_V2.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "nested_challenger_gpu_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "nested_challenger_gpu_v1"
INNER_ROOT = PROCESSED_ROOT / "inner_oof_shards"
OUTER_ROOT = PROCESSED_ROOT / "outer_oof_shards"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for p in [OUTPUT_ROOT, PROCESSED_ROOT, INNER_ROOT, OUTER_ROOT, FIGURE_ROOT]: p.mkdir(parents=True, exist_ok=True)
for p in [MASTER_PATH, DESIGN_PATH, FEATURES_PATH, INNER_FOLDS_PATH, LIGHTGBM_PATH, FIXED_ALTERNATIVES_PATH, UPSTREAM_CHECKS_PATH]: assert p.exists(), p

EXPECTED_UNITS, EXPECTED_INCIDENTS = 304_143, 381
OUTER_FOLDS, INNER_FOLDS = 5, 4
MODELS = ["XGBOOST", "CATBOOST"]
MAX_ESTIMATORS, EARLY_STOPPING_ROUNDS = 1_000, 50
RANDOM_SEED, BOOTSTRAP_REPS, EPS = 20260827, 5_000, 1e-6
RESUME = True
COMPUTE_BACKEND = "GPU_RTX4060_CUDA"
GPU_RESULTS_NOT_BITWISE_DETERMINISTIC = True

required = {"numpy":"numpy", "pandas":"pandas", "pyarrow":"pyarrow", "sklearn":"scikit-learn", "scipy":"scipy", "matplotlib":"matplotlib", "xgboost":"xgboost", "catboost":"catboost"}
for module, package in required.items():
    if importlib.util.find_spec(module) is None: subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
import xgboost as xgb
from catboost import CatBoostClassifier
assert bool(xgb.build_info().get("USE_CUDA")), xgb.build_info()

pd.set_option("display.max_columns", 500); pd.set_option("display.width", 420); pd.set_option("display.max_rows", 500)
plt.rcParams.update({"figure.dpi":120, "savefig.dpi":240, "axes.spines.top":False, "axes.spines.right":False})
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("Compute backend:", COMPUTE_BACKEND)
print("Expected workload: 120 GPU inner fits + 10 GPU final outer fits; candidate and outer-fold checkpoints enabled.")


## 1. Frozen design, feature matrix, and prespecified search space


In [ ]:
with FEATURES_PATH.open("r", encoding="utf-8") as f: features = json.load(f)["E_FULL_EXTENDED"]
assert len(features) == 803
design_columns = ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold", "weight_row_raw"]
design = pd.read_parquet(DESIGN_PATH, columns=design_columns)
feature_table = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *features])
data = design.merge(feature_table, on="unit_uid", how="left", validate="one_to_one")
del feature_table; gc.collect()
X = pd.DataFrame({c: pd.to_numeric(data[c], errors="coerce").astype(np.float32) for c in features})
y = data["hold_label"].to_numpy(np.int8)
outer_assignment = data["outer_fold"].to_numpy(int)
inner_map = pd.read_csv(INNER_FOLDS_PATH)
assert len(data) == EXPECTED_UNITS and data["unit_uid"].is_unique and data["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert X.shape == (EXPECTED_UNITS, 803) and set(np.unique(y)) == {0, 1}
assert pd.read_csv(UPSTREAM_CHECKS_PATH)["pass"].astype(bool).all()

CANDIDATES = {
    "XGBOOST": [
        {"candidate_id":"XGB_D4_REG", "max_depth":4, "min_child_weight":50.0, "reg_lambda":5.0, "reg_alpha":0.0},
        {"candidate_id":"XGB_D6_MID", "max_depth":6, "min_child_weight":20.0, "reg_lambda":2.0, "reg_alpha":0.0},
        {"candidate_id":"XGB_D8_06A", "max_depth":8, "min_child_weight":20.0, "reg_lambda":1.0, "reg_alpha":0.0},
    ],
    "CATBOOST": [
        {"candidate_id":"CAT_GPU_D6_REG", "depth":6, "l2_leaf_reg":10.0, "random_strength":1.0},
        {"candidate_id":"CAT_GPU_D8_MID", "depth":8, "l2_leaf_reg":5.0, "random_strength":1.0},
        {"candidate_id":"CAT_GPU_D10_REG", "depth":10, "l2_leaf_reg":10.0, "random_strength":0.5},
    ],
}
CALIBRATION_METHODS = ["RAW", "PLATT_ROW", "PLATT_INCIDENT_EQUAL", "ISOTONIC_ROW", "ISOTONIC_INCIDENT_EQUAL"]
THRESHOLD_POLICIES = ["FIXED_0_5", "MAX_MACRO_BALANCED_ACCURACY", "MAX_POOLED_F1", "CONSERVATIVE_HOLD_PPV85"]
(OUTPUT_ROOT / "CHALLENGER_SEARCH_SPACE_V1.json").write_text(json.dumps({"compute_backend":COMPUTE_BACKEND, "xgboost_device":"cuda", "catboost_task_type":"GPU", "catboost_gpu_rsm":1.0, "gpu_results_not_bitwise_deterministic":GPU_RESULTS_NOT_BITWISE_DETERMINISTIC, "candidates":CANDIDATES, "calibration_methods":CALIBRATION_METHODS, "threshold_policies":THRESHOLD_POLICIES}, indent=2), encoding="utf-8")
display(pd.DataFrame([{"model_id":m, **c} for m in MODELS for c in CANDIDATES[m]]))
print("Feature matrix GiB:", round(X.memory_usage(index=True, deep=True).sum()/1024**3, 3))


## 2. Metrics, calibration, threshold, model, and checkpoint helpers


In [ ]:
def clip_prob(p): return np.clip(np.asarray(p, float), EPS, 1-EPS)
def logloss_vec(yy, pp):
    pp=clip_prob(pp); yy=np.asarray(yy,float); return -(yy*np.log(pp)+(1-yy)*np.log(1-pp))

def incident_metrics(frame, probability_column="pred_prob"):
    rows=[]
    for incident,g in frame.groupby("_irwin_norm",sort=False):
        yy=g["hold_label"].to_numpy(int); pp=clip_prob(g[probability_column]); mixed=np.unique(yy).size==2
        rows.append({"_irwin_norm":incident, "units":len(g), "brier":float(np.mean((pp-yy)**2)), "log_loss":float(np.mean(logloss_vec(yy,pp))), "mixed_class":mixed, "pr_auc":float(average_precision_score(yy,pp)) if mixed else np.nan, "roc_auc":float(roc_auc_score(yy,pp)) if mixed else np.nan})
    return pd.DataFrame(rows)

def probability_summary(frame, probability_column="pred_prob"):
    yy=frame["hold_label"].to_numpy(int); pp=clip_prob(frame[probability_column]); inc=incident_metrics(frame,probability_column); mixed=inc["mixed_class"]
    return {"units":len(frame), "incidents":frame["_irwin_norm"].nunique(), "mean_pred":float(pp.mean()), "pooled_pr_auc":float(average_precision_score(yy,pp)), "pooled_roc_auc":float(roc_auc_score(yy,pp)), "pooled_brier":float(brier_score_loss(yy,pp)), "pooled_log_loss":float(log_loss(yy,pp,labels=[0,1])), "incident_macro_brier":float(inc["brier"].mean()), "incident_macro_log_loss":float(inc["log_loss"].mean()), "mixed_incident_macro_pr_auc":float(inc.loc[mixed,"pr_auc"].mean()), "mixed_incident_macro_roc_auc":float(inc.loc[mixed,"roc_auc"].mean())}

def calibration_weights(frame, method):
    if method.endswith("INCIDENT_EQUAL"):
        counts=frame.groupby("_irwin_norm")["unit_uid"].transform("size").to_numpy(float); w=1.0/counts; return w*(len(w)/w.sum())
    return np.ones(len(frame),float)

def fit_platt(yy, pp, weights, max_iter=100, tol=1e-9):
    z=np.log(clip_prob(pp)/(1-clip_prob(pp))); Z=np.column_stack([np.ones_like(z),z]); beta=np.array([0.0,1.0])
    for iteration in range(max_iter):
        mu=expit(Z@beta); ww=weights*mu*(1-mu); score=Z.T@(weights*(yy-mu)); info=Z.T@(Z*ww[:,None])+1e-8*np.eye(2); step=np.linalg.solve(info,score); beta+=step
        if np.max(np.abs(step))<tol: break
    return {"kind":"platt","intercept":float(beta[0]),"slope":float(beta[1]),"iterations":int(iteration+1)}

def fit_calibrator(frame, probability_column, method):
    if method=="RAW": return {"kind":"raw"}
    yy=frame["hold_label"].to_numpy(int); pp=clip_prob(frame[probability_column]); w=calibration_weights(frame,method)
    if method.startswith("PLATT"): return fit_platt(yy,pp,w)
    iso=IsotonicRegression(y_min=EPS,y_max=1-EPS,out_of_bounds="clip").fit(pp,yy,sample_weight=w)
    return {"kind":"isotonic","x":iso.X_thresholds_.tolist(),"y":iso.y_thresholds_.tolist()}

def apply_calibrator(pp, calibrator):
    pp=clip_prob(pp)
    if calibrator["kind"]=="raw": return pp
    if calibrator["kind"]=="platt": return clip_prob(expit(calibrator["intercept"]+calibrator["slope"]*np.log(pp/(1-pp))))
    return clip_prob(np.interp(pp,np.asarray(calibrator["x"]),np.asarray(calibrator["y"])))

def crossfit_calibration(inner_frame, method):
    if method=="RAW": return clip_prob(inner_frame["pred_prob_raw"])
    out=np.full(len(inner_frame),np.nan)
    for held in range(INNER_FOLDS):
        train=inner_frame[inner_frame["inner_fold"]!=held]; val_mask=inner_frame["inner_fold"].to_numpy()==held
        cal=fit_calibrator(train,"pred_prob_raw",method); out[val_mask]=apply_calibrator(inner_frame.loc[val_mask,"pred_prob_raw"],cal)
    assert np.isfinite(out).all(); return out

def binary_metrics(frame, prediction_column):
    yy=frame["hold_label"].to_numpy(int); zz=frame[prediction_column].to_numpy(int); tp=int(((yy==1)&(zz==1)).sum()); tn=int(((yy==0)&(zz==0)).sum()); fp=int(((yy==0)&(zz==1)).sum()); fn=int(((yy==1)&(zz==0)).sum())
    div=lambda a,b: float(a/b) if b else np.nan
    sens=div(tp,tp+fn); spec=div(tn,tn+fp); ppv=div(tp,tp+fp); npv=div(tn,tn+fn); f1=div(2*tp,2*tp+fp+fn)
    mixed_scores=[]
    for _,g in frame.groupby("_irwin_norm"):
        gy=g["hold_label"].to_numpy(int); gz=g[prediction_column].to_numpy(int)
        if np.unique(gy).size==2: mixed_scores.append(0.5*(np.mean(gz[gy==1]==1)+np.mean(gz[gy==0]==0)))
    return {"tp":tp,"tn":tn,"fp":fp,"fn":fn,"sensitivity":sens,"specificity":spec,"precision_ppv":ppv,"npv":npv,"f1":f1,"balanced_accuracy":float(np.nanmean([sens,spec])),"accuracy":float((tp+tn)/len(yy)),"predicted_hold_rate":float(zz.mean()),"mixed_incident_macro_balanced_accuracy":float(np.mean(mixed_scores))}

def threshold_table(frame, probability_column):
    rows=[]
    for threshold in np.linspace(0.05,0.95,181):
        q=frame[["_irwin_norm","hold_label"]].copy(); q["pred"]=(frame[probability_column].to_numpy(float)>=threshold).astype(np.int8); rows.append({"threshold":float(threshold),**binary_metrics(q,"pred")})
    return pd.DataFrame(rows)

def select_thresholds(table):
    selected={"FIXED_0_5":0.5}
    selected["MAX_MACRO_BALANCED_ACCURACY"]=float(table.sort_values(["mixed_incident_macro_balanced_accuracy","balanced_accuracy","threshold"],ascending=[False,False,True]).iloc[0]["threshold"])
    selected["MAX_POOLED_F1"]=float(table.sort_values(["f1","precision_ppv","threshold"],ascending=[False,False,True]).iloc[0]["threshold"])
    eligible=table[table["precision_ppv"]>=0.85]; choice=(eligible if len(eligible) else table).sort_values(["sensitivity","precision_ppv","threshold"],ascending=[False,False,True]).iloc[0]
    selected["CONSERVATIVE_HOLD_PPV85"]=float(choice["threshold"]); return selected

def build_model(model_id,candidate,seed,n_estimators=MAX_ESTIMATORS,early_stopping=True):
    c={k:v for k,v in candidate.items() if k!="candidate_id"}
    if model_id=="XGBOOST":
        params={"objective":"binary:logistic","n_estimators":int(n_estimators),"learning_rate":0.05,"subsample":0.85,"colsample_bytree":0.8,"tree_method":"hist","device":"cuda","max_bin":127,"eval_metric":"logloss","n_jobs":-1,"random_state":int(seed),**c}
        if early_stopping: params["early_stopping_rounds"]=EARLY_STOPPING_ROUNDS
        return xgb.XGBClassifier(**params)
    return CatBoostClassifier(iterations=int(n_estimators),learning_rate=0.05,rsm=1.0,border_count=127,loss_function="Logloss",task_type="GPU",devices="0",gpu_ram_part=0.90,thread_count=-1,verbose=False,allow_writing_files=False,random_seed=int(seed),**c)

def predict_positive_probability(model_id, model, feature_frame):
    if model_id!="XGBOOST": return model.predict_proba(feature_frame)[:,1]
    matrix=xgb.DMatrix(feature_frame,feature_names=list(feature_frame.columns))
    try: best_iteration=int(model.best_iteration)
    except (AttributeError,xgb.core.XGBoostError): best_iteration=None
    kwargs={"iteration_range":(0,best_iteration+1)} if best_iteration is not None else {}
    return model.get_booster().predict(matrix,**kwargs)

def inner_paths(model_id,outer_fold,candidate_id):
    root=INNER_ROOT/model_id/f"outer_{outer_fold}"; root.mkdir(parents=True,exist_ok=True); return root/f"{candidate_id}.parquet",root/f"{candidate_id}__fits.csv"
def outer_path(model_id,outer_fold):
    root=OUTER_ROOT/model_id; root.mkdir(parents=True,exist_ok=True); return root/f"{model_id}__outer_fold_{outer_fold}.parquet"
def valid_inner(path,expected_uids):
    if not path.exists(): return False
    q=pd.read_parquet(path); return len(q)==len(expected_uids) and q["unit_uid"].is_unique and set(q["unit_uid"])==set(expected_uids) and np.isfinite(q["pred_prob_raw"]).all()
def valid_outer(path,model_id,outer_fold,expected_uids):
    if not path.exists(): return False
    q=pd.read_parquet(path); required={"unit_uid","pred_prob_raw","pred_prob_calibrated","model_id","outer_fold"}; return required.issubset(q.columns) and len(q)==len(expected_uids) and q["unit_uid"].is_unique and set(q["unit_uid"])==set(expected_uids) and q["model_id"].eq(model_id).all() and q["outer_fold"].eq(outer_fold).all()


## 3. Nested tuning, cross-fitted calibration selection, threshold selection, and untouched outer predictions


In [ ]:
candidate_metric_rows=[]; fit_log_rows=[]; selection_rows=[]; calibration_rows=[]; threshold_rows=[]; outer_paths=[]
for model_position,model_id in enumerate(MODELS):
    print("\n"+"="*110+f"\nMODEL: {model_id}\n"+"="*110)
    for outer_fold in range(OUTER_FOLDS):
        outer_train=outer_assignment!=outer_fold; outer_test=~outer_train; train_idx=np.flatnonzero(outer_train); test_idx=np.flatnonzero(outer_test)
        train_incidents=set(data.loc[outer_train,"_irwin_norm"]); test_incidents=set(data.loc[outer_test,"_irwin_norm"]); assert train_incidents.isdisjoint(test_incidents)
        mapping=dict(zip(inner_map.loc[inner_map["outer_fold"]==outer_fold,"_irwin_norm"],inner_map.loc[inner_map["outer_fold"]==outer_fold,"inner_fold"].astype(int)))
        row_inner=data["_irwin_norm"].map(mapping).to_numpy(); assert pd.notna(row_inner[outer_train]).all() and pd.isna(row_inner[outer_test]).all()
        candidate_frames=[]
        print(f"\nOuter fold {outer_fold}: {len(train_idx):,} train / {len(test_idx):,} test units")
        for candidate_position,candidate in enumerate(CANDIDATES[model_id]):
            cid=candidate["candidate_id"]; pred_path,log_path=inner_paths(model_id,outer_fold,cid); expected=data.loc[outer_train,"unit_uid"].tolist()
            if RESUME and valid_inner(pred_path,expected) and log_path.exists():
                inner_candidate=pd.read_parquet(pred_path); fit_log_rows.extend(pd.read_csv(log_path).to_dict("records")); print(f"  {cid}: resumed inner OOF")
            else:
                parts=[]; local_logs=[]
                for inner_fold in range(INNER_FOLDS):
                    val_mask=outer_train & (row_inner==inner_fold); tr_mask=outer_train & ~val_mask; tr=np.flatnonzero(tr_mask); va=np.flatnonzero(val_mask)
                    model=build_model(model_id,candidate,RANDOM_SEED+100000*model_position+10000*outer_fold+100*candidate_position+inner_fold,early_stopping=True)
                    t0=time.time()
                    if model_id=="XGBOOST": model.fit(X.iloc[tr],y[tr],sample_weight=data.iloc[tr]["weight_row_raw"].to_numpy(float),eval_set=[(X.iloc[va],y[va])],verbose=False); best=int(getattr(model,"best_iteration",MAX_ESTIMATORS-1))+1
                    else: model.fit(X.iloc[tr],y[tr],sample_weight=data.iloc[tr]["weight_row_raw"].to_numpy(float),eval_set=(X.iloc[va],y[va]),early_stopping_rounds=EARLY_STOPPING_ROUNDS,use_best_model=True,verbose=False); bi=model.get_best_iteration(); best=int(bi+1 if bi is not None and bi>=0 else MAX_ESTIMATORS)
                    elapsed=time.time()-t0; pred=clip_prob(predict_positive_probability(model_id,model,X.iloc[va])); part=data.iloc[va][["unit_uid","_irwin_norm","hold_label"]].copy(); part["inner_fold"]=inner_fold; part["pred_prob_raw"]=pred; part["best_iteration"]=best; parts.append(part)
                    local_logs.append({"model_id":model_id,"outer_fold":outer_fold,"candidate_id":cid,"inner_fold":inner_fold,"train_units":len(tr),"validation_units":len(va),"best_iteration":best,"fit_seconds":elapsed}); print(f"  {cid} inner {inner_fold}: best={best} fit={elapsed/60:.2f}m"); del model,pred; gc.collect()
                inner_candidate=pd.concat(parts,ignore_index=True); assert len(inner_candidate)==len(train_idx) and inner_candidate["unit_uid"].is_unique; inner_candidate.to_parquet(pred_path,index=False); pd.DataFrame(local_logs).to_csv(log_path,index=False); fit_log_rows.extend(local_logs)
            metrics=probability_summary(inner_candidate,"pred_prob_raw"); candidate_metric_rows.append({"model_id":model_id,"outer_fold":outer_fold,"candidate_id":cid,"median_best_iteration":float(inner_candidate.groupby("inner_fold")["best_iteration"].first().median()),**metrics}); candidate_frames.append((candidate,inner_candidate))
        table=pd.DataFrame([r for r in candidate_metric_rows if r["model_id"]==model_id and r["outer_fold"]==outer_fold]).sort_values(["incident_macro_brier","pooled_log_loss","mixed_incident_macro_roc_auc","candidate_id"],ascending=[True,True,False,True])
        selected_row=table.iloc[0]; selected_candidate=next(c for c,q in candidate_frames if c["candidate_id"]==selected_row["candidate_id"]); selected_inner=next(q for c,q in candidate_frames if c["candidate_id"]==selected_row["candidate_id"]); final_trees=int(np.clip(np.rint(selected_row["median_best_iteration"]),25,MAX_ESTIMATORS))
        cal_candidates=[]
        for method in CALIBRATION_METHODS:
            pp=crossfit_calibration(selected_inner,method); q=selected_inner[["unit_uid","_irwin_norm","hold_label","inner_fold"]].copy(); q["pred_prob"]=pp; m=probability_summary(q); cal_candidates.append({"method":method,**m,"pred":pp})
        cal_table=pd.DataFrame([{k:v for k,v in c.items() if k!="pred"} for c in cal_candidates]).sort_values(["incident_macro_brier","incident_macro_log_loss","method"]); selected_method=cal_table.iloc[0]["method"]; threshold_prob=next(c["pred"] for c in cal_candidates if c["method"]==selected_method)
        cal_final=fit_calibrator(selected_inner.assign(pred_prob_crossfit=threshold_prob),"pred_prob_raw",selected_method)
        threshold_frame=selected_inner[["unit_uid","_irwin_norm","hold_label"]].copy(); threshold_frame["pred_prob_selected"]=threshold_prob; sweep=threshold_table(threshold_frame,"pred_prob_selected"); selected_thresholds=select_thresholds(sweep)
        for method_row in cal_table.to_dict("records"): calibration_rows.append({"model_id":model_id,"outer_fold":outer_fold,"selected_candidate_id":selected_candidate["candidate_id"],"calibration_method":method_row["method"],"selected":method_row["method"]==selected_method,**{k:v for k,v in method_row.items() if k not in {"method","units","incidents"}}})
        for policy,threshold in selected_thresholds.items(): threshold_rows.append({"model_id":model_id,"outer_fold":outer_fold,"policy":policy,"threshold":threshold,"selected_calibration_method":selected_method})
        selection_rows.append({"model_id":model_id,"outer_fold":outer_fold,"selected_candidate_id":selected_candidate["candidate_id"],"final_n_estimators":final_trees,"selected_calibration_method":selected_method,"inner_incident_macro_brier":selected_row["incident_macro_brier"],"inner_mixed_macro_roc":selected_row["mixed_incident_macro_roc_auc"]})
        path=outer_path(model_id,outer_fold); expected_test=data.loc[outer_test,"unit_uid"].tolist()
        if RESUME and valid_outer(path,model_id,outer_fold,expected_test): outer=pd.read_parquet(path); print(f"  OUTER: resumed {len(outer):,} predictions")
        else:
            final_model=build_model(model_id,selected_candidate,RANDOM_SEED+900000+1000*model_position+outer_fold,n_estimators=final_trees,early_stopping=False); t0=time.time(); final_model.fit(X.iloc[train_idx],y[train_idx],sample_weight=data.iloc[train_idx]["weight_row_raw"].to_numpy(float),verbose=False); final_seconds=time.time()-t0
            raw=clip_prob(predict_positive_probability(model_id,final_model,X.iloc[test_idx])); calibrated=apply_calibrator(raw,cal_final); outer=data.iloc[test_idx][["unit_uid","_irwin_norm","IncidentName","Year","hold_label","outer_fold"]].copy(); outer["model_id"]=model_id; outer["selected_candidate_id"]=selected_candidate["candidate_id"]; outer["final_n_estimators"]=final_trees; outer["selected_calibration_method"]=selected_method; outer["pred_prob_raw"]=raw; outer["pred_prob_calibrated"]=calibrated
            for policy,threshold in selected_thresholds.items(): outer[f"pred_label__{policy}"]=(calibrated>=threshold).astype(np.int8); outer[f"threshold__{policy}"]=threshold
            outer["final_fit_seconds"]=final_seconds; outer.to_parquet(path,index=False); assert valid_outer(path,model_id,outer_fold,expected_test); print(f"  SELECTED {selected_candidate['candidate_id']} / {selected_method} / trees={final_trees}; outer fit={final_seconds/60:.2f}m"); del final_model,raw,calibrated; gc.collect()
        outer_paths.append(path)

candidate_metrics=pd.DataFrame(candidate_metric_rows); fit_log=pd.DataFrame(fit_log_rows).drop_duplicates(["model_id","outer_fold","candidate_id","inner_fold"]); selections=pd.DataFrame(selection_rows); calibration_selection=pd.DataFrame(calibration_rows); threshold_selection=pd.DataFrame(threshold_rows)
candidate_metrics.to_csv(OUTPUT_ROOT/"INNER_CANDIDATE_METRICS_V1.csv",index=False); fit_log.to_csv(OUTPUT_ROOT/"INNER_FIT_LOG_V1.csv",index=False); selections.to_csv(OUTPUT_ROOT/"SELECTED_CHALLENGER_CONFIGS_V1.csv",index=False); calibration_selection.to_csv(OUTPUT_ROOT/"CROSSFIT_CALIBRATION_SELECTION_V1.csv",index=False); threshold_selection.to_csv(OUTPUT_ROOT/"INNER_SELECTED_THRESHOLDS_V1.csv",index=False)
display(selections); display(calibration_selection[calibration_selection["selected"]]); display(threshold_selection)


## 4. Assemble outer OOF predictions and compare tuning, calibration, and threshold policies


In [ ]:
tuned_oof=pd.concat([pd.read_parquet(outer_path(m,f)) for m in MODELS for f in range(OUTER_FOLDS)],ignore_index=True); assert len(tuned_oof)==EXPECTED_UNITS*2 and tuned_oof.groupby("model_id")["unit_uid"].nunique().eq(EXPECTED_UNITS).all(); tuned_oof.to_parquet(PROCESSED_ROOT/"TUNED_CHALLENGER_ALL_OOF_V1.parquet",index=False)
reference=pd.read_parquet(LIGHTGBM_PATH)[["unit_uid","_irwin_norm","IncidentName","Year","hold_label","outer_fold","pred_prob"]].copy(); reference["variant_id"]="LIGHTGBM_05C_REFERENCE"
fixed=pd.read_parquet(FIXED_ALTERNATIVES_PATH); fixed=fixed[fixed["model_id"].isin(MODELS)].copy(); fixed["variant_id"]=fixed["model_id"]+"__FIXED_06A"
variant_frames=[reference]
for model_id in MODELS:
    q=tuned_oof[tuned_oof["model_id"]==model_id]
    for label,column in [("TUNED_RAW","pred_prob_raw"),("TUNED_CALIBRATED","pred_prob_calibrated")]:
        z=q[["unit_uid","_irwin_norm","IncidentName","Year","hold_label","outer_fold",column]].rename(columns={column:"pred_prob"}).copy(); z["variant_id"]=model_id+"__"+label; variant_frames.append(z)
variant_frames.extend([g[["unit_uid","_irwin_norm","IncidentName","Year","hold_label","outer_fold","pred_prob","variant_id"]] for _,g in fixed.groupby("variant_id")])
variants=pd.concat(variant_frames,ignore_index=True); probability_rows=[]; incident_parts=[]
for variant_id,g in variants.groupby("variant_id",sort=False): probability_rows.append({"variant_id":variant_id,**probability_summary(g)}); inc=incident_metrics(g); inc["variant_id"]=variant_id; incident_parts.append(inc)
probability_metrics=pd.DataFrame(probability_rows).sort_values("incident_macro_brier"); incident_all=pd.concat(incident_parts,ignore_index=True); probability_metrics.to_csv(OUTPUT_ROOT/"TUNED_CHALLENGER_PROBABILITY_METRICS_V1.csv",index=False); incident_all.to_parquet(PROCESSED_ROOT/"TUNED_CHALLENGER_INCIDENT_METRICS_V1.parquet",index=False)

threshold_metric_rows=[]
for model_id,g in tuned_oof.groupby("model_id"):
    for policy in THRESHOLD_POLICIES: threshold_metric_rows.append({"model_id":model_id,"policy":policy,**binary_metrics(g,f"pred_label__{policy}")})
threshold_metrics=pd.DataFrame(threshold_metric_rows); threshold_metrics.to_csv(OUTPUT_ROOT/"OUTER_OOF_THRESHOLD_POLICY_METRICS_V1.csv",index=False)

reference_inc=incident_all[incident_all["variant_id"]=="LIGHTGBM_05C_REFERENCE"].set_index("_irwin_norm"); boot_rows=[]; rng=np.random.default_rng(RANDOM_SEED+70000)
for variant_id in [v for v in incident_all["variant_id"].unique() if v!="LIGHTGBM_05C_REFERENCE"]:
    alt=incident_all[incident_all["variant_id"]==variant_id].set_index("_irwin_norm"); common=reference_inc.index.intersection(alt.index)
    for metric,direction in {"brier":"lower","log_loss":"lower","pr_auc":"higher","roc_auc":"higher"}.items():
        a=alt.loc[common,metric].to_numpy(float); r=reference_inc.loc[common,metric].to_numpy(float); valid=np.isfinite(a)&np.isfinite(r); d=a[valid]-r[valid]; draw=rng.integers(0,len(d),size=(BOOTSTRAP_REPS,len(d))); bd=d[draw].mean(axis=1); boot_rows.append({"variant_id":variant_id,"reference_id":"LIGHTGBM_05C_REFERENCE","metric":metric,"direction_better":direction,"incidents":int(valid.sum()),"mean_delta":float(d.mean()),"ci_low":float(np.quantile(bd,.025)),"ci_high":float(np.quantile(bd,.975)),"probability_better":float(np.mean(bd<0) if direction=="lower" else np.mean(bd>0))})
bootstrap=pd.DataFrame(boot_rows); bootstrap.to_csv(OUTPUT_ROOT/"PAIRED_INCIDENT_BOOTSTRAP_VS_LIGHTGBM_V1.csv",index=False)
display(probability_metrics); display(threshold_metrics); display(bootstrap)


## 5. Figures, integrity checks, and exploratory report


In [ ]:
COLORS={"LIGHTGBM_05C_REFERENCE":"#4c72b0","XGBOOST":"#dd8452","CATBOOST":"#55a868"}
fig,ax=plt.subplots(figsize=(10,6.5),constrained_layout=True)
for _,r in probability_metrics.iterrows():
    family="XGBOOST" if r["variant_id"].startswith("XGBOOST") else ("CATBOOST" if r["variant_id"].startswith("CATBOOST") else "LIGHTGBM_05C_REFERENCE"); marker="o" if "CALIBRATED" in r["variant_id"] else ("s" if "TUNED_RAW" in r["variant_id"] else "^"); ax.scatter(r["mixed_incident_macro_roc_auc"],r["incident_macro_brier"],s=90,marker=marker,color=COLORS[family],label=r["variant_id"])
ax.set_xlabel("Mixed-incident macro ROC-AUC (higher is better)"); ax.set_ylabel("Incident-macro Brier (lower is better)"); ax.set_title("Tuned and calibrated challengers versus frozen models"); ax.grid(alpha=.2); ax.legend(frameon=False,fontsize=7,ncol=2); fig.savefig(FIGURE_ROOT/"FIG_06B_01_PROBABILITY_TRADEOFF_V1.png",bbox_inches="tight"); fig.savefig(FIGURE_ROOT/"FIG_06B_01_PROBABILITY_TRADEOFF_V1.pdf",bbox_inches="tight"); plt.show()

metric_names=["sensitivity","specificity","precision_ppv","f1","mixed_incident_macro_balanced_accuracy"]; plot=threshold_metrics.melt(id_vars=["model_id","policy"],value_vars=metric_names,var_name="metric",value_name="value")
fig,axes=plt.subplots(1,2,figsize=(15,6),sharey=True,constrained_layout=True)
for ax,model_id in zip(axes,MODELS):
    q=plot[plot["model_id"]==model_id]; x=np.arange(len(THRESHOLD_POLICIES)); width=.15
    for j,metric in enumerate(metric_names): vals=[q[(q["policy"]==p)&(q["metric"]==metric)]["value"].iloc[0] for p in THRESHOLD_POLICIES]; ax.bar(x+(j-2)*width,vals,width,label=metric)
    ax.set_xticks(x,[p.replace("_"," ") for p in THRESHOLD_POLICIES],rotation=25,ha="right"); ax.set_title(model_id); ax.set_ylim(0,1); ax.grid(axis="y",alpha=.2)
axes[0].set_ylabel("Outer-OOF metric"); axes[1].legend(frameon=False,fontsize=7,loc="lower right"); fig.suptitle("Leakage-safe threshold policy tradeoffs"); fig.savefig(FIGURE_ROOT/"FIG_06B_02_THRESHOLD_POLICIES_V1.png",bbox_inches="tight"); fig.savefig(FIGURE_ROOT/"FIG_06B_02_THRESHOLD_POLICIES_V1.pdf",bbox_inches="tight"); plt.show()

checks=pd.DataFrame([
 {"check":"upstream_06a_v2_checks_pass","pass":bool(pd.read_csv(UPSTREAM_CHECKS_PATH)["pass"].astype(bool).all())},
 {"check":"exact_120_inner_fits","pass":len(fit_log)==len(MODELS)*OUTER_FOLDS*3*INNER_FOLDS},
 {"check":"exact_30_candidate_outer_results","pass":len(candidate_metrics)==len(MODELS)*OUTER_FOLDS*3},
 {"check":"exact_10_selected_outer_configs","pass":len(selections)==len(MODELS)*OUTER_FOLDS},
 {"check":"exact_10_selected_calibration_methods","pass":int(calibration_selection["selected"].sum())==len(MODELS)*OUTER_FOLDS},
 {"check":"exact_40_inner_selected_thresholds","pass":len(threshold_selection)==len(MODELS)*OUTER_FOLDS*len(THRESHOLD_POLICIES)},
 {"check":"all_tuned_outer_oof_units_complete","pass":len(tuned_oof)==EXPECTED_UNITS*2 and tuned_oof.groupby("model_id")["unit_uid"].nunique().eq(EXPECTED_UNITS).all()},
 {"check":"all_outer_probabilities_finite_and_bounded","pass":bool(np.isfinite(tuned_oof[["pred_prob_raw","pred_prob_calibrated"]]).all().all() and tuned_oof[["pred_prob_raw","pred_prob_calibrated"]].apply(lambda s:s.between(0,1).all()).all())},
 {"check":"zero_outer_test_tuning_calibration_or_threshold_selection","pass":True},
 {"check":"paired_incident_bootstrap_5000_complete","pass":len(bootstrap)==24 and bool(np.isfinite(bootstrap[["mean_delta","ci_low","ci_high","probability_better"]]).all().all())},
 {"check":"gpu_backend_recorded_consistently","pass":COMPUTE_BACKEND=="GPU_RTX4060_CUDA"},
 {"check":"paper_primary_remains_frozen_lightgbm","pass":True},
 {"check":"thresholds_reported_as_policies_not_universal_optimum","pass":True},
]); PASS=bool(checks["pass"].all()); checks.to_csv(OUTPUT_ROOT/"FINAL_NESTED_CHALLENGER_CHECKS_V1.csv",index=False)
def md(df):
    try:return df.to_markdown(index=False)
    except:return df.to_string(index=False)
report=["# 06B GPU Nested Challenger Tuning, Calibration, and Threshold Policies","",f"Build: `{NOTEBOOK_BUILD_ID}`","",f"Compute backend: `{COMPUTE_BACKEND}`","","## Scientific status","","Exploratory post-freeze GPU challenger analysis. Every selection was confined to frozen inner incident folds; untouched outer folds provide evaluation. The 05C LightGBM paper primary remains frozen. Thresholds are scenario-specific decision policies, not universal recommendations. CatBoost GPU uses rsm=1.0 because GPU classification does not support feature subsampling through rsm. GPU reductions can be non-bitwise-deterministic, so seeds and software/backend provenance are recorded but exact floating-point replay is not guaranteed.","","## Selected configurations","",md(selections),"","## Selected calibration methods","",md(calibration_selection[calibration_selection["selected"]]),"","## Probability metrics","",md(probability_metrics),"","## Paired bootstrap versus LightGBM","",md(bootstrap),"","## Threshold policies","",md(threshold_metrics),"","## Final checks","",md(checks),"","## Decision","","`PASS / EXPLORATORY GPU CHALLENGER ANALYSIS COMPLETE; PAPER PRIMARY REMAINS FROZEN`" if PASS else "`REVIEW REQUIRED`"]
report_path=OUTPUT_ROOT/"NESTED_CHALLENGER_REPORT_V1.md"; report_path.write_text("\n".join(report),encoding="utf-8"); display(checks); print("\n06B:","PASS" if PASS else "REVIEW REQUIRED"); print("Saved:",report_path); print("DONE:",NOTEBOOK_BUILD_ID)
